In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
!mkdir -p /content/drive/MyDrive/hm_project
%cd /content/drive/MyDrive/hm_project

/content/drive/MyDrive/hm_project


In [40]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (3).json


{'kaggle (3).json': b'{"username":"afnakurikkal22","key":"63bd3eb4964ab63e60ba1567e965f17e"}'}

In [41]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [42]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f articles.csv

100% 4.26M/4.26M [00:00<00:00, 65.1MB/s]



In [43]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f customers.csv

100% 97.9M/97.9M [00:01<00:00, 89.5MB/s]



In [44]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f transactions_train.csv

100% 584M/584M [00:06<00:00, 99.2MB/s]



In [45]:
!unzip -o -q "*.zip"
!rm *.zip


3 archives were successfully processed.


In [46]:
!ls -lh

total 3.5G
-rw------- 1 root root  35M Jan 17  2022  articles.csv
drwx------ 2 root root 4.0K Nov  5 06:44  customer_feats.parquet
-rw------- 1 root root 198M Jan 17  2022  customers.csv
drwx------ 2 root root 4.0K Nov  5 06:45  item_popularity.parquet
-rw------- 1 root root   70 Nov  4 17:21 'kaggle (1).json'
-rw------- 1 root root   70 Nov  5 05:52 'kaggle (2).json'
-rw------- 1 root root   70 Nov  5 07:00 'kaggle (3).json'
-rw------- 1 root root   70 Nov  4 11:45  kaggle.json
-rw------- 1 root root 3.3G Jan 17  2022  transactions_train.csv


In [47]:
import pandas as pd
articles = pd.read_csv('articles.csv')
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [48]:
!pip install -q pyspark

In [49]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("hm_project") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [50]:
transactions = spark.read.csv(
    "transactions_train.csv",
    header=True,
    inferSchema=True
)

In [51]:
!ls -lh /content

total 8.0K
drwx------ 5 root root 4.0K Nov  5 05:52 drive
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [52]:
transactions.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [53]:
transactions.show(5, truncate=False)

+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|t_dat     |customer_id                                                     |article_id|price               |sales_channel_id|
+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|663713001 |0.050830508474576264|2               |
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|541518023 |0.03049152542372881 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|505221004 |0.01523728813559322 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|685687003 |0.016932203389830508|2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|685687004 |0.016932203389830508|2 

In [54]:
print(transactions.count())

31788324


In [55]:
from pyspark.sql import functions as F

transactions.select(F.min("t_dat"), F.max("t_dat")).show()

+----------+----------+
|min(t_dat)|max(t_dat)|
+----------+----------+
|2018-09-20|2020-09-22|
+----------+----------+



In [56]:
transactions.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transactions.columns]
).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



In [57]:
print("Customers:", transactions.select("customer_id").distinct().count())
print("Articles:", transactions.select("article_id").distinct().count())

Customers: 1362281
Articles: 104547


In [58]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
articles = spark.read.csv("articles.csv", header=True, inferSchema=True)

In [59]:
customers.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers.columns]
).show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+



In [60]:
customers.select("age").describe().show()

+-------+-----------------+
|summary|              age|
+-------+-----------------+
|  count|          1356119|
|   mean|  36.386964565794|
| stddev|14.31362798162793|
|    min|               16|
|    max|               99|
+-------+-----------------+



In [61]:
print("Customers in customers.csv:", customers.count())
print("Articles in articles.csv:", articles.count())

Customers in customers.csv: 1371980
Articles in articles.csv: 105542


In [62]:
no_purchase = customers.join(
    transactions.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
)
print("Customers with no purchases:", no_purchase.count())

Customers with no purchases: 9699


In [63]:
unsold = articles.join(
    transactions.select("article_id").distinct(),
    on="article_id",
    how="left_anti"
)
print("Articles never bought:", unsold.count())

Articles never bought: 995


In [64]:
median_age = int(customers.approxQuantile("age", [0.5], 0.01)[0])
print("Median age:", median_age)

Median age: 32


In [65]:
customers_clean = (
    customers
    .withColumn("age_missing", F.col("age").isNull().cast("int"))
    .withColumn("fashion_news_frequency", F.upper(F.col("fashion_news_frequency")))
    .fillna({
        "age": median_age,
        "FN": 0,
        "Active": 0,
        "club_member_status": "UNKNOWN",
        "fashion_news_frequency": "NONE"
    })
)

In [66]:
customers_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers_clean.columns]
).show()

+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|customer_id| FN|Active|club_member_status|fashion_news_frequency|age|postal_code|age_missing|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|          0|  0|     0|                 0|                     0|  0|          0|          0|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+



In [67]:
customers_clean.groupBy("fashion_news_frequency").count().show()

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|               MONTHLY|   842|
|                  NONE|893722|
|             REGULARLY|477416|
+----------------------+------+



In [68]:
cutoff = "2020-09-16"
train = transactions.filter(F.col("t_dat") < cutoff)
test = transactions.filter(F.col("t_dat") >= cutoff)
print("Train rows:", train.count())
print("Test rows:", test.count())

Train rows: 31548013
Test rows: 240311


In [69]:
train = (
    train
    .withColumn("month", F.month("t_dat"))
    .withColumn("day_of_week", F.dayofweek("t_dat"))
    .withColumn("days_ago", F.datediff(F.lit(cutoff).cast("date"), F.col("t_dat")))
)

In [70]:
customer_feats = (
    train.groupBy("customer_id")
    .agg(
        F.count("*").alias("n_purchases"),
        F.countDistinct("article_id").alias("n_unique_items"),
        F.avg("price").alias("avg_price"),
        F.min("days_ago").alias("days_since_last_purchase")
    )
)
customer_feats.show(5)

+--------------------+-----------+--------------+--------------------+------------------------+
|         customer_id|n_purchases|n_unique_items|           avg_price|days_since_last_purchase|
+--------------------+-----------+--------------+--------------------+------------------------+
|25add38a9584701b7...|        183|           170|0.029469852736871353|                      36|
|e38cedeb861af12e9...|         11|             5|0.023058551617873648|                     545|
|22e699f8a8d74c1ea...|          6|             6| 0.01834463276836158|                     722|
|c6e8b198cc0e4ddd6...|        299|           219| 0.04250104869338472|                       5|
|b473ffd0c6b98aadd...|         10|             6| 0.03049152542372881|                     707|
+--------------------+-----------+--------------+--------------------+------------------------+
only showing top 5 rows


In [71]:
item_popularity = (
    train.filter(F.col("days_ago") <= 28)
    .groupBy("article_id")
    .agg(F.count("*").alias("purchases_last_28d"))
)
top_items = item_popularity.orderBy(F.desc("purchases_last_28d")).limit(12)
top_items.show()

+----------+------------------+
|article_id|purchases_last_28d|
+----------+------------------+
| 751471001|              2758|
| 706016001|              2408|
| 915526001|              2383|
| 751471043|              2328|
| 918292001|              2327|
| 915529003|              2177|
| 896152002|              2028|
| 898694001|              1836|
| 916468003|              1822|
| 448509014|              1807|
| 863595006|              1767|
| 915526002|              1600|
+----------+------------------+



In [72]:
customer_feats.write.mode("overwrite").parquet("customer_feats.parquet")
item_popularity.write.mode("overwrite").parquet("item_popularity.parquet")